# Week 3 - Day 1 - Hands On Lab
Setting Up the ML Workflow (Features, Target, Train/Test Split)

## Objective

I'm working with the Auto MPG dataset. My goal today is not to build a model yet — it's to set up a clean, correct workflow: understand the dataset, clean it using the EDA habits I built in Weeks 1 and 2, then separate it into features (X) and target (y), and split it into training and testing sets.

This workflow is the foundation every supervised learning project is built on, and I'll reuse it tomorrow when I train my first regression model.

## Import Libraries

In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split

## Load Data

The Auto MPG dataset records specs for 398 cars from the 1970s-80s, along with their fuel efficiency (`mpg`). It's a classic dataset for practicing regression.

In [39]:
df = pd.read_csv("data/auto-mpg.csv")
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


## Step 1: Understand the Dataset

Before touching anything, I want to know what I'm working with — how many rows and columns, and what each column means.

| Column | Meaning |
|---|---|
| mpg | Fuel efficiency (miles per gallon) — this is my target |
| cylinders | Number of engine cylinders |
| displacement | Engine displacement (cubic inches) |
| horsepower | Engine horsepower |
| weight | Car weight (lbs) |
| acceleration | Time to accelerate (seconds) |
| model year | Year the model was released |
| origin | Country code (1 = USA, 2 = Europe, 3 = Japan) |
| car name | Car make/model — an identifier, not something the model should learn from |

In [40]:
print(df.shape)
df.info()

(398, 9)
<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    398 non-null    str    
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model year    398 non-null    int64  
 7   origin        398 non-null    int64  
 8   car name      398 non-null    str    
dtypes: float64(3), int64(4), str(2)
memory usage: 28.1 KB


**What this shows:** 398 rows, 9 columns. All columns show "non-null" — at first glance it looks like there's no missing data. But `horsepower` is typed as `str` (text) instead of a number, which is suspicious for a column that should be purely numeric. This is exactly the kind of thing I learned to check for during EDA in Week 2 — a column's dtype can hint at hidden problems even when `.isnull()` doesn't.

In [41]:
df[df["horsepower"] == "?"]

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
32,25.0,4,98.0,?,2046,19.0,71,1,ford pinto
126,21.0,6,200.0,?,2875,17.0,74,1,ford maverick
330,40.9,4,85.0,?,1835,17.3,80,2,renault lecar deluxe
336,23.6,4,140.0,?,2905,14.3,80,1,ford mustang cobra
354,34.5,4,100.0,?,2320,15.8,81,2,renault 18i
374,23.0,4,151.0,?,3035,20.5,82,1,amc concord dl


**What this shows:** the horsepower column contains a `?` value. That's a missing value disguised as text instead of an actual `NaN`, which is why `.isnull()` didn't catch it earlier.

---

## Step 2: Clean the Data

I'll convert `horsepower` to a proper numeric column. Any `?` will become a real `NaN` that pandas can recognize, using `errors="coerce"`.

In [42]:
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")
df["horsepower"].isnull().sum()

np.int64(6)

In [43]:
missing_pct = df["horsepower"].isna().mean() * 100
print(f"Missing values: {missing_pct:.2f}%")

Missing values: 1.51%


**What this shows:** There are 6 missing values in horsepower, representing only 1.51% of the dataset. Since this is a very small portion of the data, removing these rows is unlikely to affect the analysis or model performance.

In [44]:
df = df.dropna(subset=["horsepower"])
df.shape

(392, 9)

In [45]:
df.info()

<class 'pandas.DataFrame'>
Index: 392 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           392 non-null    float64
 1   cylinders     392 non-null    int64  
 2   displacement  392 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        392 non-null    int64  
 5   acceleration  392 non-null    float64
 6   model year    392 non-null    int64  
 7   origin        392 non-null    int64  
 8   car name      392 non-null    str    
dtypes: float64(4), int64(4), str(1)
memory usage: 30.6 KB


**What this shows:** The dataset now contains 392 complete rows, and horsepower has been successfully converted to a numeric (float64) column with no missing values remaining.

---

## Step 3: Drop Columns That Aren't Features

`car name` is a text identifier — 305 different values across 392 rows. It doesn't describe a measurable property of the car, so the model can't learn anything useful from it. I'll drop it before separating features and target.

In [46]:
df = df.drop("car name", axis=1)
df.columns.tolist()

['mpg',
 'cylinders',
 'displacement',
 'horsepower',
 'weight',
 'acceleration',
 'model year',
 'origin']

---

## Step 4: Quick Sanity Check on the Target

Before splitting anything, I want a quick look at how the features relate to `mpg`, using the correlation habit from Week 2.

In [47]:
df.corr(numeric_only=True)["mpg"].sort_values()

weight         -0.832244
displacement   -0.805127
horsepower     -0.778427
cylinders      -0.777618
acceleration    0.423329
origin          0.565209
model year      0.580541
mpg             1.000000
Name: mpg, dtype: float64

**What this shows:** `weight`, `displacement`, `horsepower`, and `cylinders` all have a strong negative correlation with `mpg` — heavier cars with bigger engines are less fuel-efficient, which matches common sense. This is a good sign the dataset has real, learnable patterns for tomorrow's regression model.

**Note:** `origin` is actually a country code (1/2/3), not a true continuous number — I'm leaving it as-is for today since today's focus is the X/y and train/test workflow, not feature engineering. I'll revisit how to handle categorical-looking numeric columns like this later in the week.

---

## Step 5: Separate Features (X) and Target (y)

`mpg` is what I want to predict, so it's my target. Everything else is a feature.

In [48]:
X = df.drop("mpg", axis=1)
y = df["mpg"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (392, 7)
y shape: (392,)


In [49]:
X.head()

,cylinders,displacement,horsepower,weight,acceleration,model year,origin
0,8,307.0,130.0,3504,12.0,70,1
1,8,350.0,165.0,3693,11.5,70,1
2,8,318.0,150.0,3436,11.0,70,1
3,8,304.0,150.0,3433,12.0,70,1
4,8,302.0,140.0,3449,10.5,70,1


In [50]:
y.head()

0    18.0
1    15.0
2    18.0
3    16.0
4    17.0
Name: mpg, dtype: float64

**What this shows:** 7 features and 392 rows, matching the target's 392 values — X and y are aligned correctly.

---

## Step 6: Train/Test Split

>At this point, the dataset is fully cleaned and organized. The features and target are ready to be split into training and testing sets for supervised learning.

My dataset is small-to-medium sized (392 rows), so I'm using an 80/20 split — enough data to train on, and enough held-out rows to trust the evaluation.

In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (313, 7)
X_test: (79, 7)
y_train: (313,)
y_test: (79,)


In [52]:
X_train.head()

,cylinders,displacement,horsepower,weight,acceleration,model year,origin
260,6,225.0,110.0,3620,18.7,78,1
184,4,140.0,92.0,2572,14.9,76,1
174,6,171.0,97.0,2984,14.5,75,1
64,8,318.0,150.0,4135,13.5,72,1
344,4,86.0,64.0,1875,16.4,81,1


**What this shows:** 313 rows for training and 79 for testing — roughly an 80/20 split. `random_state=42` means this exact split will be reproducible every time I rerun this notebook.

---

## Why the Model Must Never See X_test During Training

If I trained the model on all 392 rows and then evaluated it on some of those same rows, I wouldn't actually be testing whether it learned anything — I'd just be checking whether it memorized the answers. That would make the model look better than it really is.

By holding out `X_test` and `y_test` completely until evaluation time, I get an honest answer to the real question: can this model make good predictions on cars it has never seen before? That's the whole point of splitting the data before training even starts.

---

## Summary

- Loaded the Auto MPG dataset (398 rows, 9 columns)
- Found a hidden data quality issue: `horsepower` had missing values disguised as `"?"` instead of `NaN`
- Cleaned the data: converted `horsepower` to numeric, dropped 6 incomplete rows, dropped the non-feature `car name` column
- Checked correlations with `mpg` as a sanity check before modeling
- Separated the cleaned data into features (X, 7 columns) and target (y, `mpg`)
- Split the data into training (313 rows) and testing (79 rows) sets with `random_state=42` for reproducibility

The dataset is now ready for training my first regression model tomorrow.